# 🧘 AntarJyoti User Feedback Database Viewer
This notebook provides simple SQL commands to query and analyze the user feedback telemetry stored inside the SQLite database (`chroma.sqlite3`).

In [ ]:
import os
import sqlite3
import pandas as pd

# Locate the project root folder dynamically by searching upwards for the .git directory
def get_project_root():
    path = os.path.abspath(os.getcwd())
    while path != os.path.dirname(path):
        if '.git' in os.listdir(path):
            return path
        path = os.path.dirname(path)
    # Fallback to current working directory
    return os.path.abspath(os.getcwd())

root_dir = get_project_root()
print(f"📁 Resolved project root directory: {root_dir}")

# Setup absolute paths to the SQLite database files
db_path = os.path.join(root_dir, 'chroma_prod.sqlite3')

if not os.path.exists(db_path):
    db_path = os.path.join(root_dir, 'backend', 'chroma_db_backup', 'chroma.sqlite3')
if not os.path.exists(db_path):
    db_path = os.path.join(root_dir, 'backend', 'chroma_db_ollama', 'chroma.sqlite3')

if os.path.exists(db_path):
    print(f"✅ Connected to database file at: {db_path}")
    
    # Ensure the user_feedback table exists so queries do not throw an error
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS user_feedback (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            question TEXT,
            answer TEXT,
            sources TEXT,
            feedback_value INTEGER,
            latency_ms REAL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        );
    """)
    conn.commit()
    conn.close()
    print("ℹ️ Table 'user_feedback' verified / created.")
else:
    print(f"❌ Could not find database file. Looked at: {db_path}")

### 💡 Tip: How to view your live Fly.io feedback logs
If you want to view the ratings you submitted on the live website, run this command in your terminal to download the production database file to your computer:

```bash
rm -f chroma_prod.sqlite3 && fly sftp get -a antarjyoti-backend /data/chroma_db_backup/chroma.sqlite3 ./chroma_prod.sqlite3
```
Once downloaded, re-run this notebook and it will automatically connect to the downloaded production database (`chroma_prod.sqlite3`).

## 📊 1. Fetch All Feedback Data
Loads the feedback log into a Pandas DataFrame.

In [ ]:
conn = sqlite3.connect(db_path)

# Load feedback table to Pandas DataFrame
query = "SELECT id, question, feedback_value, latency_ms, timestamp FROM user_feedback ORDER BY timestamp DESC;"
df = pd.read_sql_query(query, conn)
conn.close()

# Render the table
df

## 📈 2. Aggregate Feedback & Latency Statistics
Summarizes helpfulness scores and response times.

In [ ]:
conn = sqlite3.connect(db_path)

stats_df = pd.read_sql_query("""
    SELECT 
        COUNT(*) as total_responses,
        SUM(case when feedback_value = 1 then 1 else 0 end) as helpful_count,
        SUM(case when feedback_value = -1 then 1 else 0 end) as unhelpful_count,
        ROUND(AVG(latency_ms) / 1000.0, 2) as avg_latency_seconds,
        ROUND(MIN(latency_ms) / 1000.0, 2) as min_latency_seconds,
        ROUND(MAX(latency_ms) / 1000.0, 2) as max_latency_seconds
    FROM user_feedback;
""", conn)

conn.close()
stats_df

## 🔍 3. Custom SQL Query Sandbox
You can modify the query below to inspect the full entries, answers, and sources.

In [ ]:
conn = sqlite3.connect(db_path)

# Example: Fetch only negative feedback to see what needs improvement
custom_query = """
    SELECT question, answer, latency_ms 
    FROM user_feedback 
    WHERE feedback_value = -1;
"""

df_custom = pd.read_sql_query(custom_query, conn)
conn.close()
df_custom